In [ ]:
from Forecast.SimpleTransformerForecast import SimpleTransformerForecast
from Dataloaders.DataloaderECG import ECGDataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from tqdm.notebook import tqdm
import torch

In [ ]:
# Funcion para entrenar el modelo
def train_loop(model, train, val, optimizer, scheduler=None, patience=5, epochs=100, lossf=F.mse_loss):
    """_Bucle de entrenamiento_

    Args:
        model: red a entrenar
        optimizer: optimizador de pytorch, por ejemplo torch.optim.Adam
        train: datos de entrenamiento
        val: datos de validacion
        epochs: numero de epochs

    Returns:
        _type_: _description_
    """
    def epoch_loss(dataset):
        data_loss = 0.0
        for i, (data, labels) in enumerate(dataset):
            inputs = data.to('cuda')
            y = labels.to('cuda')
            outputs = model(inputs)
            loss = lossf(y, outputs)
            data_loss += loss.item()  
        return data_loss / i  
    
    def early_stopping(val_loss, patience=5):
        if len(val_loss) > patience:
            if val_loss[-1] > np.mean(val_loss[-(patience+1):-1]):
                return True
    
    hist_loss = {'train': [], 'val': []}
    pbar = tqdm(range(epochs))
    for epoch in pbar:  # bucle para todos los epochs
        for i, (data, labels) in enumerate(train):
            # obtenemos los datos y los subimos a la GPU
            inputs = data.to('cuda')
            y = labels.to('cuda')

            # Reiniciamos los gradientes
            optimizer.zero_grad()

            # Aplicamos los datos al modelo
            outputs = model(inputs)
            # Calculamos la perdida
            loss = lossf(y, outputs)

            # Hacemos el paso hacia atras
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        # Calculamos la perdida en el conjunto de entrenamiento y validacion
        with torch.no_grad():
            hist_loss['train'].append(epoch_loss(train))
            hist_loss['val'].append(epoch_loss(val))

        # Mostramos la perdida en el conjunto de entrenamiento y validacion
        pbar.set_postfix({'train': hist_loss['train'][-1], 'val': hist_loss['val'][-1]})

        # Si la perdida en el conjunto de validacion no disminuye, paramos el entrenamiento
        if early_stopping(hist_loss['val'], patience):
            break
            
    return hist_loss 

In [ ]:
data = ECGDataset(dir='../Preprocess/PTBXL', dataset='train', nsamples=100, channel=0, lookback=120, horizon=10, stride=10)

In [ ]:
model = SimpleTransformerForecast(input_dim=1, d_model=64, n_heads=8, n_layers=4, target_dim=1, horizon=10, n_channels=1)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)